In [0]:
## Unity Catalog context (Azure)
spark.sql('USE CATALOG projeto_data_engineering')
spark.sql('USE SCHEMA kabum_project')

# Storage account name 
STORAGE_ACCOUNT = 'storagekabumdata'
BRONZE_ABFSS = f"abfss://bronze@{STORAGE_ACCOUNT}.dfs.core.windows.net/"
SILVER_ABFSS = f"abfss://silver@{STORAGE_ACCOUNT}.dfs.core.windows.net/"
GOLD_ABFSS   = f"abfss://gold@{STORAGE_ACCOUNT}.dfs.core.windows.net/"


In [0]:
%sql
-- GOLD_FEATURES_V2_SCORED_PATH
-- /Volumes/workspace/kabum_project/lake/gold/kabum/notebooks_features_enriched_scored_delta

-- GOLD_QUALITY_DAILY_PATH
-- /Volumes/workspace/kabum_project/lake/gold/kabum/catalog_quality_daily_delta

Views para o dashboard

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW v_kabum_features_scored AS
SELECT *
FROM projeto_data_engineering.kabum_project.notebooks_features_scored;

CREATE OR REPLACE TEMP VIEW v_kabum_quality_daily AS
SELECT *
FROM projeto_data_engineering.kabum_project.catalog_quality_daily;

KPIs do último dia disponível

In [0]:
%sql
WITH last_day AS (
  SELECT max(ingestion_date) AS ingestion_date
  FROM v_kabum_quality_daily
)
SELECT
  q.ingestion_date,
  q.search_term,
  q.marketplace,
  q.products,
  q.avg_completeness_pct,
  q.median_completeness_pct,
  q.high_pct,
  q.medium_pct,
  q.low_pct
FROM v_kabum_quality_daily q
JOIN last_day d
  ON q.ingestion_date = d.ingestion_date
ORDER BY q.search_term, q.marketplace;

KPIs filtráveis por período (para “trend cards”)

In [0]:
%sql
SELECT
  ingestion_date,
  search_term,
  marketplace,
  products,
  avg_completeness_pct,
  median_completeness_pct,
  high_pct,
  medium_pct,
  low_pct
FROM v_kabum_quality_daily
ORDER BY ingestion_date DESC, search_term, marketplace;

Avg completeness ao longo do tempo

In [0]:
%sql
SELECT
  ingestion_date,
  search_term,
  marketplace,
  avg_completeness_pct
FROM v_kabum_quality_daily
ORDER BY ingestion_date, search_term, marketplace;

Stacked % tiers ao longo do tempo

In [0]:
%sql
SELECT
  ingestion_date,
  search_term,
  marketplace,
  high_pct,
  medium_pct,
  low_pct
FROM v_kabum_quality_daily
ORDER BY ingestion_date, search_term, marketplace;

Distribuição de score (histograma)

In [0]:
%sql
SELECT
  catalog_completeness_score,
  count(*) AS products
FROM v_kabum_features_scored
GROUP BY catalog_completeness_score
ORDER BY catalog_completeness_score;

Fill-rate por coluna (qualidade por spec)

In [0]:
%sql
WITH base AS (
  SELECT
    price,
    old_price,
    discount_pct,
    rating,
    reviews_count,
    refresh_rate_hz_scraped,
    screen_resolution_scraped,
    panel_type_scraped,
    panel_finish_scraped,
    brightness_nits_scraped,
    product_url_scraped
  FROM v_kabum_features_scored
)
SELECT 'price' AS field,
       AVG(CASE WHEN price IS NOT NULL THEN 1.0 ELSE 0.0 END) AS fill_rate
FROM base
UNION ALL
SELECT 'old_price',
       AVG(CASE WHEN old_price IS NOT NULL THEN 1.0 ELSE 0.0 END)
FROM base
UNION ALL
SELECT 'discount_pct',
       AVG(CASE WHEN discount_pct IS NOT NULL THEN 1.0 ELSE 0.0 END)
FROM base
UNION ALL
SELECT 'rating',
       AVG(CASE WHEN rating IS NOT NULL THEN 1.0 ELSE 0.0 END)
FROM base
UNION ALL
SELECT 'reviews_count',
       AVG(CASE WHEN reviews_count IS NOT NULL THEN 1.0 ELSE 0.0 END)
FROM base
UNION ALL
SELECT 'refresh_rate_hz_scraped',
       AVG(CASE WHEN refresh_rate_hz_scraped IS NOT NULL THEN 1.0 ELSE 0.0 END)
FROM base
UNION ALL
SELECT 'screen_resolution_scraped',
       AVG(CASE WHEN screen_resolution_scraped IS NOT NULL THEN 1.0 ELSE 0.0 END)
FROM base
UNION ALL
SELECT 'panel_type_scraped',
       AVG(CASE WHEN panel_type_scraped IS NOT NULL THEN 1.0 ELSE 0.0 END)
FROM base
UNION ALL
SELECT 'panel_finish_scraped',
       AVG(CASE WHEN panel_finish_scraped IS NOT NULL THEN 1.0 ELSE 0.0 END)
FROM base
UNION ALL
SELECT 'brightness_nits_scraped',
       AVG(CASE WHEN brightness_nits_scraped IS NOT NULL THEN 1.0 ELSE 0.0 END)
FROM base
UNION ALL
SELECT 'product_url_scraped',
       AVG(CASE WHEN product_url_scraped IS NOT NULL THEN 1.0 ELSE 0.0 END)
FROM base;

Produtos com LOW completeness (top exemplos)

In [0]:
%sql
SELECT
  product_name,
  brand,
  price,
  rating,
  reviews_count,
  catalog_completeness_score,
  catalog_completeness_pct,
  catalog_completeness_tier,
  refresh_rate_hz_scraped,
  screen_resolution_scraped,
  panel_type_scraped,
  panel_finish_scraped,
  brightness_nits_scraped,
  product_url
FROM v_kabum_features_scored
WHERE catalog_completeness_tier = 'LOW'
ORDER BY catalog_completeness_pct ASC, product_name
LIMIT 50;


Gamer sem refresh_rate (flag)

In [0]:
%sql
SELECT
  product_name,
  brand,
  price,
  search_term,
  refresh_rate_hz_scraped,
  screen_resolution_scraped,
  panel_type_scraped,
  product_url
FROM v_kabum_features_scored
WHERE lower(search_term) LIKE '%gamer%'
  AND refresh_rate_hz_scraped IS NULL
ORDER BY product_name
LIMIT 50;


Fonte do painel (TITLE/RAW/MISSING)

In [0]:
%sql
SELECT
  panel_type_scraped AS panel_type,
  COUNT(*) AS products,
  ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 2) AS pct
FROM v_kabum_features_scored
GROUP BY panel_type_scraped
ORDER BY products DESC;


Distribuição por Tipo de Painel (Market Mix)

In [0]:
%sql
SELECT
  panel_type_scraped AS panel_type,
  COUNT(*) AS products,
  ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 2) AS pct
FROM v_kabum_features_scored
GROUP BY panel_type_scraped
ORDER BY products DESC;


Distribuição de Refresh Rate

In [0]:
%sql
SELECT
  refresh_rate_hz_scraped AS refresh_rate_hz,
  COUNT(*) AS products
FROM v_kabum_features_scored
GROUP BY refresh_rate_hz_scraped
ORDER BY refresh_rate_hz_scraped;


Gamer vs Não Gamer

In [0]:
%sql
SELECT
  CASE 
    WHEN lower(product_name) RLIKE '(gamer|aorus|nitro|legion|predator|tuf|rog|strix|omen|alienware)'
    THEN 'GAMER'
    ELSE 'NON_GAMER'
  END AS segment,
  COUNT(*) AS products
FROM v_kabum_features_scored
GROUP BY segment;

Distribuição de Tamanho de Tela

In [0]:
%sql
SELECT
  category,
  COUNT(*) AS products
FROM v_kabum_features_scored
GROUP BY category
ORDER BY products DESC
LIMIT 20;


Top CPUs (Market Intelligence)

In [0]:
%sql
SELECT
  search_term,
  COUNT(*) AS products
FROM v_kabum_features_scored
GROUP BY search_term
ORDER BY products DESC;


Completeness vs Categoria Gamer

In [0]:
%sql
SELECT
  CASE 
    WHEN lower(product_name) RLIKE '(gamer|aorus|nitro|legion|predator|tuf|rog|strix|omen|alienware)'
    THEN 'GAMER'
    ELSE 'NON_GAMER'
  END AS segment,
  AVG(catalog_completeness_pct) AS avg_completeness
FROM v_kabum_features_scored
GROUP BY segment;

Distribuição de Storage

In [0]:
%sql
SELECT
  CAST(brightness_nits_scraped AS DOUBLE) AS brightness_nits,
  COUNT(*) AS products
FROM v_kabum_features_scored
WHERE brightness_nits_scraped IS NOT NULL
GROUP BY CAST(brightness_nits_scraped AS DOUBLE)
ORDER BY brightness_nits;


KPI executivo do “último dia”

In [0]:
%sql
WITH last_day AS (
  SELECT max(ingestion_date) AS ingestion_date
  FROM v_kabum_quality_daily
)
SELECT
  q.ingestion_date,
  q.search_term,
  q.marketplace,
  q.products,
  q.avg_completeness_pct,
  q.median_completeness_pct,
  q.high_pct,
  q.medium_pct,
  q.low_pct
FROM v_kabum_quality_daily q
JOIN last_day d
  ON q.ingestion_date = d.ingestion_date
ORDER BY q.search_term, q.marketplace;

Marca líder do catálogo (presença)

In [0]:
%sql
SELECT
  brand,
  COUNT(*) AS products,
  ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 2) AS pct_of_catalog
FROM v_kabum_features_scored
GROUP BY brand
ORDER BY products DESC
LIMIT 20;

Ticket médio por marca (preço atual)

In [0]:
%sql
SELECT
  brand,
  COUNT(*) AS products,
  ROUND(AVG(price), 2) AS avg_price,
  ROUND(percentile_approx(price, 0.5), 2) AS median_price
FROM v_kabum_features_scored
WHERE price IS NOT NULL
GROUP BY brand
HAVING COUNT(*) >= 3
ORDER BY avg_price DESC;

“Valor de portfólio” por marca (proxy de potencial de receita)

In [0]:
%sql
SELECT
  brand,
  COUNT(*) AS products,
  ROUND(SUM(price), 2) AS total_catalog_value
FROM v_kabum_features_scored
WHERE price IS NOT NULL
GROUP BY brand
ORDER BY total_catalog_value DESC;

KPI de desconto (média e mediana por marca)

In [0]:
%sql
SELECT
  brand,
  COUNT(*) AS products,
  ROUND(AVG(discount_pct), 2) AS avg_discount_pct,
  ROUND(percentile_approx(discount_pct, 0.5), 2) AS median_discount_pct
FROM v_kabum_features_scored
WHERE discount_pct IS NOT NULL
GROUP BY brand
HAVING COUNT(*) >= 3
ORDER BY avg_discount_pct DESC;

Desconto real em R$ (melhores “deals”)

In [0]:
%sql
SELECT
  product_name,
  brand,
  ROUND(old_price, 2) AS old_price,
  ROUND(price, 2) AS price,
  ROUND(old_price - price, 2) AS discount_brl,
  ROUND(discount_pct, 2) AS discount_pct
FROM v_kabum_features_scored
WHERE price IS NOT NULL
  AND old_price IS NOT NULL
  AND old_price >= price
ORDER BY discount_brl DESC
LIMIT 50;


Segmento Gamer vs Não-Gamer (mix + ticket + desconto)

In [0]:
%sql
WITH base AS (
  SELECT
    CASE 
      WHEN lower(product_name) RLIKE '(gamer|aorus|nitro|legion|predator|tuf|rog|strix|omen|katana|victus|loq|helios|zephyrus|alienware|ideapad gaming)'
      THEN 'GAMER'
      ELSE 'NON_GAMER'
    END AS segment,
    price,
    discount_pct
  FROM v_kabum_features_scored
)
SELECT
  segment,
  COUNT(*) AS products,
  ROUND(AVG(price), 2) AS avg_price,
  ROUND(percentile_approx(price, 0.5), 2) AS median_price,
  ROUND(AVG(discount_pct), 2) AS avg_discount_pct
FROM base
GROUP BY segment
ORDER BY products DESC;

Distribuição de preço (faixas)

In [0]:
%sql
SELECT
  CASE
    WHEN price < 2000 THEN '< 2k'
    WHEN price < 3000 THEN '2k-3k'
    WHEN price < 4000 THEN '3k-4k'
    WHEN price < 5000 THEN '4k-5k'
    WHEN price < 7000 THEN '5k-7k'
    WHEN price < 10000 THEN '7k-10k'
    ELSE '>= 10k'
  END AS price_bucket,
  COUNT(*) AS products
FROM v_kabum_features_scored
WHERE price IS NOT NULL
GROUP BY price_bucket
ORDER BY products DESC;

“Premium share” por marca (ex: % acima de 6k)

In [0]:
%sql
WITH b AS (
  SELECT brand, price
  FROM v_kabum_features_scored
  WHERE price IS NOT NULL
)
SELECT
  brand,
  COUNT(*) AS products,
  SUM(CASE WHEN price >= 6000 THEN 1 ELSE 0 END) AS premium_cnt,
  ROUND(SUM(CASE WHEN price >= 6000 THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS premium_pct
FROM b
GROUP BY brand
HAVING COUNT(*) >= 3
ORDER BY premium_pct DESC;

Value For Money Score

In [0]:
%sql
SELECT
  product_name,
  brand,
  price,
  rating,
  reviews_count,
  refresh_rate_hz_scraped,
  brightness_nits_scraped,

  ROUND(
    (
      COALESCE(CAST(rating AS DOUBLE), 0.0) +
      COALESCE(CAST(refresh_rate_hz_scraped AS DOUBLE), 0.0) / 60.0 +
      COALESCE(CAST(brightness_nits_scraped AS DOUBLE), 0.0) / 300.0 +
      COALESCE(CAST(reviews_count AS DOUBLE), 0.0) / 1000.0
    ) / CAST(price AS DOUBLE)
  , 8) AS value_score

FROM v_kabum_features_scored
WHERE price IS NOT NULL
ORDER BY value_score DESC
LIMIT 20;


Criar o score de performance

In [0]:
%sql
SELECT
  product_name,
  brand,
  price,
  rating,
  reviews_count,
  refresh_rate_hz_scraped,
  brightness_nits_scraped,

  (
    COALESCE(CAST(rating AS DOUBLE), 0.0) +
    COALESCE(CAST(refresh_rate_hz_scraped AS DOUBLE), 0.0) / 60.0 +
    COALESCE(CAST(brightness_nits_scraped AS DOUBLE), 0.0) / 300.0 +
    COALESCE(CAST(reviews_count AS DOUBLE), 0.0) / 1000.0
  ) AS performance_score

FROM v_kabum_features_scored
WHERE price IS NOT NULL;


Query final para o gráfico

In [0]:
%sql
SELECT
  product_name,
  brand,
  price,
  rating,
  reviews_count,
  refresh_rate_hz_scraped,
  brightness_nits_scraped,

  (
    COALESCE(CAST(rating AS DOUBLE), 0.0) +
    COALESCE(CAST(refresh_rate_hz_scraped AS DOUBLE), 0.0) / 60.0 +
    COALESCE(CAST(brightness_nits_scraped AS DOUBLE), 0.0) / 300.0 +
    COALESCE(CAST(reviews_count AS DOUBLE), 0.0) / 1000.0
  ) AS performance_score

FROM v_kabum_features_scored
WHERE price IS NOT NULL
  AND (rating IS NOT NULL OR refresh_rate_hz_scraped IS NOT NULL OR brightness_nits_scraped IS NOT NULL)
ORDER BY performance_score DESC
LIMIT 50;


Price segmentation (Budget / Mid / Premium)

Segmentação geral

In [0]:
%sql
WITH base AS (
  SELECT
    *,
    CASE
      WHEN price < 3500 THEN 'BUDGET (<3.5k)'
      WHEN price < 7000 THEN 'MID (3.5k–7k)'
      WHEN price < 12000 THEN 'PREMIUM (7k–12k)'
      ELSE 'ULTRA (12k+)'
    END AS price_segment
  FROM v_kabum_features_scored
  WHERE price IS NOT NULL
)
SELECT
  price_segment,
  COUNT(*) AS products,
  ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) AS pct_products,
  ROUND(AVG(price), 2) AS avg_price,
  ROUND(percentile_approx(price, 0.5), 2) AS median_price
FROM base
GROUP BY price_segment
ORDER BY
  CASE price_segment
    WHEN 'BUDGET (<3.5k)' THEN 1
    WHEN 'MID (3.5k–7k)' THEN 2
    WHEN 'PREMIUM (7k–12k)' THEN 3
    ELSE 4
  END;

Segmentação por marca (market positioning)

In [0]:
%sql
WITH base AS (
  SELECT
    brand,
    price,
    CASE
      WHEN price < 3500 THEN 'BUDGET'
      WHEN price < 7000 THEN 'MID'
      WHEN price < 12000 THEN 'PREMIUM'
      ELSE 'ULTRA'
    END AS price_segment
  FROM v_kabum_features_scored
  WHERE price IS NOT NULL
)
SELECT
  brand,
  price_segment,
  COUNT(*) AS products,
  ROUND(AVG(price),2) AS avg_price
FROM base
GROUP BY brand, price_segment
ORDER BY brand, price_segment;

brand,price_segment,products,avg_price
ASUS,BUDGET,16,2973.71
ASUS,MID,18,5266.82
ASUS,PREMIUM,8,9096.24
ASUS,ULTRA,20,29314.7
Acer,BUDGET,4,3199.05
Acer,MID,38,5205.06
Acer,PREMIUM,8,7902.7
Acer,ULTRA,6,14999.1
Apple,PREMIUM,2,11999.0
Dell,MID,2,3689.1


Brand market share (share do catálogo)

In [0]:
%sql
SELECT
  brand,
  COUNT(*) AS products,
  ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) AS market_share_pct,
  ROUND(AVG(price),2) AS avg_price,
  ROUND(percentile_approx(price,0.5),2) AS median_price
FROM v_kabum_features_scored
WHERE brand IS NOT NULL
GROUP BY brand
ORDER BY products DESC;

Share por segmento (quem domina Budget/Mid/Premium)

In [0]:
%sql
WITH base AS (
  SELECT
    brand,
    price,
    CASE
      WHEN price < 3500 THEN 'BUDGET'
      WHEN price < 7000 THEN 'MID'
      WHEN price < 12000 THEN 'PREMIUM'
      ELSE 'ULTRA'
    END AS price_segment
  FROM v_kabum_features_scored
  WHERE price IS NOT NULL AND brand IS NOT NULL
),
seg_tot AS (
  SELECT price_segment, COUNT(*) AS seg_total
  FROM base
  GROUP BY price_segment
)
SELECT
  b.price_segment,
  b.brand,
  COUNT(*) AS products,
  ROUND(COUNT(*) * 100.0 / t.seg_total, 2) AS share_within_segment_pct
FROM base b
JOIN seg_tot t USING (price_segment)
GROUP BY b.price_segment, b.brand, t.seg_total
ORDER BY b.price_segment, products DESC;